# 005: Finding innovation-culture sentences with embeddings and Gemini Flash

**Hands-on Exercise 1 — Frontier LLMs and RAG**

This notebook demonstrates a simplified classroom version of **Li, Mai, Shen, Yang & Zhang (2026)**, focusing only on **innovation and adaptability culture**.

The exercise is designed to be read step by step. Each stage explains both the coding task and the measurement idea behind it: how we move from raw earnings-call text to a smaller set of sentences that may capture a culture construct.

## Background: Li et al. (2026) and this classroom simplification

Li et al. (2026) use a modern hybrid workflow to measure corporate culture from text. The key idea is **culture-first filtering**:

1. Identify text segments that are truly about **corporate or organizational culture**.
2. Use semantic/classifier methods to recover culture discussion that does not contain exact culture keywords.
3. Use generative AI to filter false positives and extract structured culture information.
4. Use RAG when the focal text segment needs more surrounding context.

The Internet Appendix is especially useful for this exercise. It shows that their first screen is not simply a search for business words such as `innovation`, `technology`, or `change`. Instead, they first ask whether a segment is really about corporate culture, using definitions such as:

- principles and values that guide employee behavior;
- shared beliefs, assumptions, values, or preferences that drive group behavior;
- norms and values widely shared and strongly held throughout the organization;
- informal institutions reflected in behavioral patterns reinforced by events, people, and systems.

Only after identifying culture-related text do they classify culture type. This notebook keeps the exercise simple and focuses on one culture type: **innovation and adaptability**.

### Classroom workflow

This notebook implements a simplified version:

1. **Keyword retrieval:** search for explicit corporate-culture phrases and paper-inspired innovation/adaptability culture-type phrases.
2. **Semantic retrieval:** use sentence embeddings to find sentences close to the innovation/adaptability culture concept.
3. **Gemini Flash filtering:** ask Gemini to first decide whether the sentence is about corporate culture, and then whether the culture type is innovation/adaptability.

### Definition: innovation and adaptability culture

Following the Li et al. taxonomy, innovation and adaptability culture means corporate culture focusing on, or deficient in, **innovation, creativity, technology, entrepreneurship, adaptability, transformation, flexibility, agility, willingness to experiment, going beyond tradition, disruption, being fast-moving, quickly taking advantage of opportunities, resilience to change, or taking initiative**.

### Important distinction

A sentence is **not** innovation/adaptability culture merely because it mentions:

- a new product;
- technology, AI, software, or R&D;
- innovation spending or digital transformation;
- market disruption or growth opportunities;
- a new business strategy.

The sentence must discuss these ideas as part of **corporate culture, organizational norms, values, practices, routines, leadership approach, employee behavior, or ways of working**.

**Pedagogical message:** retrieval is part of measurement construction. If the retrieval step starts with the wrong keywords, the measured construct will drift away from the intended culture concept.


## Setup: packages, paths, and API access

This section prepares the notebook for GitHub and Colab use. The package installation cell makes the notebook easier to run on a new machine, while the path variables keep all output files in an `outputs/` folder.

The input CSV is loaded directly from the GitHub `raw` branch, so students do not need to upload the data manually in Colab.

The Gemini API key is read from Colab Secrets using `userdata.get("Gemini_API_Key")`. Outside Colab, the same code falls back to the local environment variable `Gemini_API_Key`. We do not hard-code API keys in notebooks because notebooks are often shared with students or uploaded to GitHub.


In [ ]:
# Quiet package installation (no GPU required)
import subprocess
import sys

packages = [
    "pandas", "numpy", "scikit-learn", "sentence-transformers",
    "tqdm", "google-genai", "json-repair", "matplotlib",
]
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q"] + packages
)

In [ ]:
import json
import os
import re
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from google import genai
from google.genai import types
from json_repair import repair_json
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm

try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    userdata = None
    IN_COLAB = False

GITHUB_RAW_BASE = (
    "https://raw.githubusercontent.com/"
    "helenlu-vbs/NLP_LLM_for_Finance_and-Accounting_Research-Sheffield-/raw"
)
INPUT_FILE = f"{GITHUB_RAW_BASE}/003_all_US_calls_2024Q4_top500.csv"

# In Colab, outputs are saved to /content by default. Locally, use the teaching folder.
PROJECT_DIR = Path("/content") if IN_COLAB else Path(r"C:\Users\Helen\Dropbox\Sheffield_NLP_teaching")
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_SENTENCES = 5000
MAX_LLM_SENTENCES = 60  # Set MAX_LLM_SENTENCES = 20 for a quick classroom demo
BATCH_SIZE = 10
TEMPERATURE = 0
MODEL_NAME = "gemini-3.5-flash"


CORPORATE_CULTURE_PHRASES = [
    "corporate culture",
    "company culture",
    "company's culture",
    "firm culture",
    "firm's culture",
    "organizational culture",
    "workplace culture",
    "business culture",
    "culture in the company",
]

# Paper-inspired examples for the Innovation and Adaptability culture type
# from Li, Mai, Shen, Yang & Zhang (2026), Internet Appendix Table IA4.
INNOVATION_ADAPTABILITY_CULTURE_PHRASES = [
    "innovative culture",
    "entrepreneurial culture",
    "growth-oriented culture",
    "growth oriented culture",
    "innovative corporate culture",
    "data-driven culture",
    "data driven culture",
    "technology-driven culture",
    "technology driven culture",
    "innovation-driven culture",
    "innovation driven culture",
    "knowledge-driven culture",
    "knowledge driven culture",
    "creative and innovative culture",
    "entrepreneurial and decentralized culture",
    "adaptive culture",
    "adaptive corporate culture",
    "change-oriented culture",
    "change oriented culture",
    "resilient culture",
    "proactive culture",
    "continuous improvement culture",
    "evolving culture",
    "transformative culture",
    "transformational culture",
    "agile culture",
]

# Do not use these as standalone keyword hits. They are too broad in earnings calls.
BROAD_CONTEXT_WORDS_NOT_STANDALONE_KEYWORDS = [
    "innovation", "innovative", "technology", "AI", "digital", "transformation",
    "R&D", "growth", "change", "adaptability", "agility", "flexibility",
    "experimentation", "initiative", "creativity",
]

KEYWORD_PHRASES = CORPORATE_CULTURE_PHRASES + INNOVATION_ADAPTABILITY_CULTURE_PHRASES
KEYWORD_PATTERNS = [
    (phrase, re.compile(r"\b" + re.escape(phrase) + r"\b", re.IGNORECASE))
    for phrase in KEYWORD_PHRASES
]


def get_gemini_api_key():
    if userdata is not None:
        try:
            key = userdata.get("Gemini_API_Key")
            if key:
                return key
        except Exception:
            pass
    return os.getenv("Gemini_API_Key")


api_key = get_gemini_api_key()
client = genai.Client(api_key=api_key) if api_key else None

print(f"Running in Colab: {IN_COLAB}")
print(f"Input file: {INPUT_FILE}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Gemini_API_Key set: {bool(api_key)}")

## 1. Load earnings-call turns

We start with the raw unit available in the CSV: an earnings-call **turn**. A turn is one speaker contribution, not yet a sentence. Loading the data first lets us inspect the columns and confirm that the text field we will analyze is `turn_text`.

This step also removes empty turns, because missing text cannot contribute to either keyword search, embeddings, or LLM filtering.

In [ ]:
df = pd.read_csv(INPUT_FILE)
print("Shape:", df.shape)
print("Columns:", list(df.columns))
display(df.head())

df["call_date"] = pd.to_datetime(df["call_date"], errors="coerce")
before = len(df)
df = df.dropna(subset=["turn_text"]).copy()
print(f"Dropped {before - len(df)} rows with missing turn_text. Remaining: {len(df)}")

## 2. Split turns into sentences

Most culture claims are easier to evaluate at the **sentence** level than at the full-turn level. A full answer may contain several topics, while a sentence usually gives a cleaner unit for retrieval and validation.

We keep the call and speaker metadata so that every sentence can still be traced back to the company, call date, speaker, and original transcript component. The length filter removes very short fragments and very long passages that are less useful for a classroom demonstration.

In [ ]:
METADATA_COLS = [
    "companyname", "companyid", "transcriptid", "call_date", "headline",
    "turn_index", "transcriptcomponentid", "transcriptcomponenttypeid",
    "component_type", "speaker_name", "speaker_type",
]

SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")


def split_into_sentences(text: str) -> list[str]:
    text = str(text).strip()
    if not text:
        return []
    parts = SENTENCE_SPLIT_RE.split(text)
    return [p.strip() for p in parts if p.strip()]


def word_count(s: str) -> int:
    return len(re.findall(r"\b\w+\b", s))


rows = []
sent_counter = 0
for _, row in tqdm(df.iterrows(), total=len(df), desc="Splitting turns"):
    for sentence in split_into_sentences(row["turn_text"]):
        sent_counter += 1
        rec = {col: row[col] for col in METADATA_COLS}
        rec["sent_id"] = sent_counter
        rec["sentence"] = sentence
        rows.append(rec)

sentences_df = pd.DataFrame(rows)
print(f"Raw sentences: {len(sentences_df):,}")

sentences_df = sentences_df.drop_duplicates(subset=["sentence"]).copy()
sentences_df["n_words"] = sentences_df["sentence"].map(word_count)
sentences_df = sentences_df[
    (sentences_df["n_words"] >= 8) & (sentences_df["n_words"] <= 120)
].copy()
print(f"After dedup + length filter (8–120 words): {len(sentences_df):,}")

# Preserve all exact culture/culture-type phrase hits before applying the classroom cap.
# This keeps Stage 1 visible even when exact culture phrases are rare in a random sample.
def has_exact_culture_phrase(sentence: str) -> bool:
    return any(pat.search(sentence) for _, pat in KEYWORD_PATTERNS)

sentences_df["_exact_keyword_preserve"] = sentences_df["sentence"].map(has_exact_culture_phrase)

if len(sentences_df) > MAX_SENTENCES:
    exact_hits = sentences_df[sentences_df["_exact_keyword_preserve"]]
    non_hits = sentences_df[~sentences_df["_exact_keyword_preserve"]]
    n_non_hits = max(0, MAX_SENTENCES - len(exact_hits))
    if len(non_hits) > n_non_hits:
        non_hits = non_hits.sample(n=n_non_hits, random_state=42)
    sentences_df = pd.concat([exact_hits, non_hits], ignore_index=True).sort_values("sent_id")
    print(
        f"Sampled to MAX_SENTENCES={MAX_SENTENCES}, preserving "
        f"{len(exact_hits):,} exact culture keyword hits"
    )

sentences_df = sentences_df.drop(columns=["_exact_keyword_preserve"])

sentences_df = sentences_df.reset_index(drop=True)
display(sentences_df.head())

## Stage 1: Culture-first keyword retrieval

The first revision from the paper is conceptual: do **not** start by searching for broad business words such as `innovation`, `technology`, `AI`, `digital`, or `change`. Those words are common in earnings calls and often describe products, strategy, spending, or markets rather than culture.

Instead, this notebook uses two conservative keyword groups:

1. **Corporate-culture phrases** from the Li et al. Internet Appendix, such as `corporate culture`, `company culture`, and `organizational culture`.
2. **Innovation/adaptability culture-type phrases** inspired by their taxonomy examples, such as `innovative culture`, `entrepreneurial culture`, `adaptive culture`, `continuous improvement culture`, and `agile culture`.

This keyword step is intentionally narrow. It gives students transparent, auditable hits while leaving room for semantic retrieval to find implicit culture discussion.


In [ ]:
def match_keywords(sentence: str) -> tuple[bool, str]:
    matched = [phrase for phrase, pat in KEYWORD_PATTERNS if pat.search(sentence)]
    return bool(matched), "; ".join(matched)


hits = sentences_df["sentence"].map(match_keywords)
sentences_df["innovation_keyword_hit"] = hits.map(lambda x: x[0])
sentences_df["innovation_matched_keywords"] = hits.map(lambda x: x[1])

n_hits = int(sentences_df["innovation_keyword_hit"].sum())
pct_hits = 100 * n_hits / len(sentences_df)
print(f"Keyword hits: {n_hits:,} ({pct_hits:.2f}% of sentences)")
print("\nKeyword design note:")
print("- Broad words not used as standalone hits:")
print(", ".join(BROAD_CONTEXT_WORDS_NOT_STANDALONE_KEYWORDS))

print("\n--- 15 example keyword-hit sentences ---")
examples = sentences_df[sentences_df["innovation_keyword_hit"]].head(15)
for _, row in examples.iterrows():
    print(f"\n[{row['innovation_matched_keywords']}] {row['sentence'][:300]}")


**Reflection:** Which keyword hits are truly about corporate culture? Which merely contain the word `culture` without telling us whether the culture is innovation/adaptability? Why did we avoid broad standalone words such as `innovation`, `technology`, and `change`?


## Stage 2: Semantic retrieval

Semantic retrieval asks a different question from keyword search: not “does this sentence contain one of our exact phrases?” but “is this sentence close in meaning to the innovation/adaptability culture concept?”

The query below is deliberately anchored in **culture language**. This matters because a query about only `innovation`, `technology`, or `transformation` would retrieve many ordinary product and strategy sentences. The goal is to retrieve sentences about organizational culture, norms, behaviors, capabilities, and ways of working.


In [ ]:
SEMANTIC_QUERY = (
    "corporate culture or organizational culture characterized by innovative culture, "
    "entrepreneurial culture, adaptive culture, agile culture, resilient culture, "
    "proactive culture, continuous improvement culture, transformative culture, "
    "technology-driven culture, data-driven culture, willingness to experiment, "
    "openness to change, fast-moving organization, quickly taking advantage of opportunities, "
    "resilience to change, taking initiative, and new ways of working"
)

embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
sentence_list = sentences_df["sentence"].tolist()
sentence_embeddings = embed_model.encode(
    sentence_list, show_progress_bar=True, batch_size=64
)
query_embedding = embed_model.encode([SEMANTIC_QUERY])
sentences_df["innovation_semantic_score"] = cosine_similarity(
    query_embedding, sentence_embeddings
)[0]

top200_ids = set(
    sentences_df.nlargest(200, "innovation_semantic_score")["sent_id"].tolist()
)
semantic_mask = (
    sentences_df["sent_id"].isin(top200_ids)
    | (sentences_df["innovation_semantic_score"] >= 0.35)
)
candidate_mask = sentences_df["innovation_keyword_hit"] | semantic_mask

innovation_candidates_df = sentences_df[candidate_mask].copy()


def candidate_reason(row) -> str:
    kw = bool(row["innovation_keyword_hit"])
    sem = row["sent_id"] in top200_ids or row["innovation_semantic_score"] >= 0.35
    if kw and sem:
        return "both"
    if kw:
        return "keyword_only"
    return "semantic_only"


innovation_candidates_df["candidate_reason"] = innovation_candidates_df.apply(
    candidate_reason, axis=1
)

print("Candidate counts by reason:")
print(innovation_candidates_df["candidate_reason"].value_counts())


def show_examples(reason: str, n: int = 10):
    print(f"\n=== {reason} (n={n}) ===")
    subset = innovation_candidates_df[
        innovation_candidates_df["candidate_reason"] == reason
    ].head(n)
    for _, row in subset.iterrows():
        print(f"\nscore={row['innovation_semantic_score']:.3f} | {row['sentence'][:280]}")


for r in ["keyword_only", "semantic_only", "both"]:
    show_examples(r, 10)

reason_counts = innovation_candidates_df["candidate_reason"].value_counts()
plt.figure(figsize=(6, 4))
reason_counts.plot(kind="bar", color=["#4C72B0", "#DD8452", "#55A868"])
plt.title("Innovation/adaptability culture candidates by retrieval reason")
plt.xlabel("candidate_reason")
plt.ylabel("Number of sentences")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


**Semantic-only examples are the key teaching moment:** they show what exact culture keywords would have missed. But they also need careful filtering, because semantic similarity can still retrieve strategy, product, or technology sentences that are not culture measurement.


## Stage 3: Gemini Flash filtering

The retrieval steps intentionally cast a wider net than the final construct. Following the logic of Li et al. (2026), Gemini is used as a **culture filter and type classifier**.

Gemini checks a balanced sample from all retrieval sources: `keyword_only`, `semantic_only`, and `both`. This means it audits exact culture-phrase hits as well as embedding-retrieved sentences.

The prompt asks Gemini to make two decisions:

1. Is the sentence truly about corporate or organizational culture?
2. If yes, is the culture type specifically innovation and adaptability?

This is more conservative than asking only whether a sentence mentions innovation. It should reduce false positives about new products, AI, R&D, digital strategy, market disruption, or growth opportunities.


In [ ]:
INNOVATION_CULTURE_FILTER_PROMPT = """
You are helping with an academic textual-analysis exercise inspired by Li, Mai, Shen, Yang & Zhang (2026) on corporate culture.

Task:
For each earnings-call sentence below, make two decisions:
1. Does the sentence substantively discuss corporate or organizational culture?
2. If it is about corporate or organizational culture, is the culture type Innovation and Adaptability?

Corporate or organizational culture means any of the following:
- principles and values that guide employee behavior;
- shared beliefs, assumptions, values, or preferences that drive group behavior;
- norms and values that are widely shared and strongly held throughout the organization;
- an informal institution typified by patterns of behavior and reinforced by events, people, and systems.

Innovation and Adaptability culture means culture focusing on, or deficient in, innovation, creativity, technology, entrepreneurship, adaptability, transformation, flexibility, agility, willingness to experiment, going beyond tradition, disruption, being fast-moving, quickly taking advantage of opportunities, resilience to change, or taking initiative.

Important exclusion rule:
Do NOT classify a sentence as Innovation and Adaptability culture merely because it mentions:
- a new product;
- technology, AI, software, or R&D;
- innovation spending;
- digital transformation;
- market disruption;
- growth opportunities;
- a new business strategy;
- operational initiatives.

These topics count only if the sentence connects them to corporate culture, organizational norms, values, practices, routines, leadership approach, employee behavior, capabilities, or ways of working.

Examples that should be classified as Innovation and Adaptability culture:
- "We have built an innovative culture that encourages experimentation."
- "Our teams are adopting more agile ways of working across the organization."
- "The company has become more entrepreneurial and quicker to take initiative."
- "Management is encouraging a proactive culture of continuous improvement."

Examples that should NOT be classified as Innovation and Adaptability culture:
- "We launched a new AI product."
- "R&D spending increased this quarter."
- "Digital revenue grew by 20%."
- "The market is experiencing disruption."
- "We are investing in technology to support growth."

Return strict JSON only.
Return a list with one object per input sentence.

Each object must have:
- sent_id
- is_corporate_culture: true or false
- is_innovation_adaptability_culture: true or false
- explicit_or_implicit: explicit, implicit, or not_culture
- confidence: number between 0 and 1
- evidence_phrase: a short exact phrase from the sentence, or "" if not culture-related
- reason: one concise sentence explaining the decision

Sentences:
{sentences_json}
"""


def balanced_gemini_sample(candidates: pd.DataFrame, max_n: int) -> pd.DataFrame:
    reasons = ["keyword_only", "semantic_only", "both"]
    per_group = max(1, max_n // len(reasons))
    parts = []
    for reason in reasons:
        subset = candidates[candidates["candidate_reason"] == reason]
        if len(subset) == 0:
            continue
        n_take = min(per_group, len(subset))
        parts.append(subset.sample(n=n_take, random_state=42))
    if not parts:
        return candidates.head(0).copy()
    sample = pd.concat(parts).drop_duplicates(subset=["sent_id"])
    if len(sample) < max_n:
        remaining = candidates[~candidates["sent_id"].isin(sample["sent_id"])]
        if len(remaining) > 0:
            fill_n = min(max_n - len(sample), len(remaining))
            fill = remaining.sample(n=fill_n, random_state=42)
            sample = pd.concat([sample, fill]).drop_duplicates(subset=["sent_id"])
    if len(sample) > max_n:
        sample = sample.sample(n=max_n, random_state=42)
    return sample.sort_values("sent_id").reset_index(drop=True)


def parse_gemini_json(text: str) -> list:
    text = text.strip()
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        parsed = json.loads(repair_json(text))
    if isinstance(parsed, dict):
        if "all_results" in parsed:
            parsed = parsed["all_results"]
        elif "results" in parsed:
            parsed = parsed["results"]
    if not isinstance(parsed, list):
        raise ValueError(f"Expected JSON list, got {type(parsed)}")
    return parsed


def run_gemini_batch(batch_df: pd.DataFrame, max_retries: int = 3) -> list:
    payload = [
        {"sent_id": int(row.sent_id), "sentence": row.sentence}
        for row in batch_df.itertuples(index=False)
    ]
    prompt = INNOVATION_CULTURE_FILTER_PROMPT.format(
        sentences_json=json.dumps(payload, ensure_ascii=False)
    )

    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=TEMPERATURE,
                    response_mime_type="application/json",
                ),
            )
            return parse_gemini_json(response.text)
        except Exception as exc:
            last_error = exc
            wait_seconds = 10 * attempt
            print(
                f"Gemini call failed on attempt {attempt}/{max_retries}: {exc}. "
                f"Waiting {wait_seconds} seconds before retrying."
            )
            time.sleep(wait_seconds)

    raise RuntimeError(f"Gemini failed after {max_retries} attempts") from last_error


if client is None:
    raise ValueError(
        "Set Gemini_API_Key in Colab Secrets (left sidebar > key icon), "
        "or set a local environment variable named Gemini_API_Key before running Stage 3."
    )

gemini_input_df = balanced_gemini_sample(innovation_candidates_df, MAX_LLM_SENTENCES)
print(f"Gemini input sample: {len(gemini_input_df)} sentences")
print(gemini_input_df["candidate_reason"].value_counts())

gemini_results = []
for start in tqdm(range(0, len(gemini_input_df), BATCH_SIZE), desc="Gemini batches"):
    batch = gemini_input_df.iloc[start : start + BATCH_SIZE]
    gemini_results.extend(run_gemini_batch(batch))

gemini_df = pd.DataFrame(gemini_results)
gemini_df = gemini_df.rename(
    columns={
        "is_corporate_culture": "gemini_is_corporate_culture",
        "is_innovation_adaptability_culture": "gemini_is_innovation_adaptability_culture",
        "explicit_or_implicit": "gemini_explicit_or_implicit",
        "confidence": "gemini_confidence",
        "evidence_phrase": "gemini_evidence_phrase",
        "reason": "gemini_reason",
    }
)
gemini_df["sent_id"] = gemini_df["sent_id"].astype(int)

innovation_gemini_df = gemini_input_df.merge(gemini_df, on="sent_id", how="left")
display(innovation_gemini_df.head())


## Save outputs

The outputs are saved in an `outputs/` folder. The candidate file captures the retrieval stage; the Gemini-filtered file captures the small LLM-scored sample; and the manual validation file is designed for classroom audit and discussion.


In [ ]:
sentences_out = OUTPUT_DIR / "005_innovation_sentences.csv"
candidates_out = OUTPUT_DIR / "005_innovation_candidates.csv"
gemini_out = OUTPUT_DIR / "005_innovation_gemini_filtered.csv"
validation_out = OUTPUT_DIR / "005_innovation_manual_validation_sample.csv"

sentences_df.to_csv(sentences_out, index=False)
innovation_candidates_df.to_csv(candidates_out, index=False)
innovation_gemini_df.to_csv(gemini_out, index=False)

print(f"Saved: {sentences_out}")
print(f"Saved: {candidates_out}")
print(f"Saved: {gemini_out}")

## Manual validation sample

The Gemini labels are useful, but they are not automatically a research variable. A researcher still needs to inspect a validation sample and compare model labels with human judgment.

This sample is deliberately balanced across retrieval sources and Gemini outcomes. That makes it easier to see where the method performs well and where it fails, instead of only reviewing the most obvious cases.

In [ ]:
def build_manual_validation_sample(gemini_scored: pd.DataFrame, n: int = 30) -> pd.DataFrame:
    scored = gemini_scored.dropna(subset=["gemini_is_innovation_adaptability_culture"]).copy()
    if scored.empty:
        return pd.DataFrame()

    scored["gemini_is_innovation_adaptability_culture"] = scored[
        "gemini_is_innovation_adaptability_culture"
    ].map(lambda x: str(x).lower() in {"true", "1", "yes"})

    buckets = {
        "keyword_only": scored[scored["candidate_reason"] == "keyword_only"],
        "semantic_only": scored[scored["candidate_reason"] == "semantic_only"],
        "both": scored[scored["candidate_reason"] == "both"],
        "gemini_positive": scored[scored["gemini_is_innovation_adaptability_culture"]],
        "gemini_negative": scored[~scored["gemini_is_innovation_adaptability_culture"]],
    }
    per_bucket = max(1, n // len(buckets))
    picked = []
    for name, subset in buckets.items():
        if len(subset) == 0:
            continue
        take = subset.sample(n=min(per_bucket, len(subset)), random_state=42)
        take = take.copy()
        take["validation_bucket"] = name
        picked.append(take)
    sample = pd.concat(picked).drop_duplicates(subset=["sent_id"])
    if len(sample) < n:
        remaining = scored[~scored["sent_id"].isin(sample["sent_id"])]
        if len(remaining) > 0:
            fill_n = min(n - len(sample), len(remaining))
            fill = remaining.sample(n=fill_n, random_state=42).copy()
            fill["validation_bucket"] = "fill_remaining"
            sample = pd.concat([sample, fill]).drop_duplicates(subset=["sent_id"])
    if len(sample) > n:
        sample = sample.sample(n=n, random_state=42)

    cols = [
        "sent_id", "sentence", "companyname", "call_date", "headline",
        "speaker_name", "speaker_type", "component_type",
        "innovation_keyword_hit", "innovation_matched_keywords",
        "innovation_semantic_score", "candidate_reason",
        "gemini_is_corporate_culture", "gemini_is_innovation_adaptability_culture",
        "gemini_explicit_or_implicit", "gemini_confidence",
        "gemini_evidence_phrase", "gemini_reason",
    ]
    sample = sample[[c for c in cols if c in sample.columns]].copy()
    sample["human_label_blank"] = ""
    sample["notes_blank"] = ""
    return sample.sort_values("sent_id").reset_index(drop=True)


validation_df = build_manual_validation_sample(innovation_gemini_df, n=30)
validation_df.to_csv(validation_out, index=False)
print(f"Saved: {validation_out} ({len(validation_df)} rows)")
display(validation_df.head())

## Gemini results visualization

The final chart gives a quick sense of how many retrieved candidates survive the LLM filter. This is not a final accuracy estimate, but it helps students see how much filtering happens after retrieval.

The high-confidence positive examples are useful for discussion because they show the kind of language the model treats as clear evidence of innovation and adaptability culture after first checking that the sentence is about corporate or organizational culture.


In [ ]:
gemini_scored = innovation_gemini_df.dropna(
    subset=["gemini_is_innovation_adaptability_culture"]
).copy()
gemini_scored["gemini_positive"] = gemini_scored[
    "gemini_is_innovation_adaptability_culture"
].map(lambda x: str(x).lower() in {"true", "1", "yes"})

counts = gemini_scored["gemini_positive"].value_counts().rename(
    index={True: "true innovation/adaptability culture", False: "not culture"}
)
plt.figure(figsize=(6, 4))
counts.plot(kind="bar", color=["#55A868", "#C44E52"])
plt.title("Gemini classification counts")
plt.ylabel("Number of sentences")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

top_positive = (
    gemini_scored[gemini_scored["gemini_positive"]]
    .sort_values("gemini_confidence", ascending=False)
    .head(10)[["sent_id", "sentence", "gemini_confidence", "gemini_evidence_phrase", "gemini_reason"]]
)
print("Top 10 highest-confidence Gemini-positive sentences:")
display(top_positive)

## Class discussion: from retrieval to measurement

1. Which exact keyword hits are about corporate culture but not specifically innovation/adaptability culture?
2. Which semantic-only examples look genuinely culture-related?
3. Did semantic retrieval find examples that exact culture phrase search missed?
4. Where does Gemini confuse strategy, products, or technology with culture?
5. Are Gemini's explanations useful for auditing the output?
6. What manual validation would be required before using this as a research variable?
7. How is this classroom workflow similar to and different from Li et al. (2026)?


## What we learned

- Culture measurement should start by identifying text that is truly about corporate or organizational culture.
- Broad standalone words such as `innovation`, `technology`, and `change` create many false positives in earnings calls.
- Exact culture phrases are transparent but narrow.
- Embeddings help find implicit culture discussion that exact phrase search misses.
- Frontier LLMs can filter candidate passages and explain why they are, or are not, culture-related.
- Model outputs are not research variables until validated.
- This notebook demonstrates a first step in modern textual-analysis measurement: retrieving candidate text for construct construction.
